# Phase 4 — distill a decoder-only LoRMA CJPE teacher into a smaller decoder


In [1]:
# Run once, then restart the kernel if huggingface_hub was already imported.
%pip install -q --upgrade "huggingface_hub==2.2.0" tokenizers scikit-learn tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.2.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
diffusers 0.40.0 requir

In [2]:
import os, json, math, random, gc, tempfile, zipfile, uuid, re, hashlib
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfFileSystem
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
BUCKET_ROOT = 'hf://buckets/A-240031/legal-lm-phase1-bucket'
PHASE1_TAG = 'decoder_126M'
MODEL_PATH = 'checkpoints/decoder_126M/model.pt'
TOKENIZER_PATH = 'tokenizer_126M_decoder/tokenizer.json'
MODEL_CONFIG = dict(d_model=1024, n_heads=8, n_kv_heads=4, n_layers=8, d_ff=2816, max_seq_len=2048)
MAX_SEQ_LEN = 2048
CHUNK_OVERLAP = 64  # number of shared tokens, NOT the stride
CHUNK_ENCODE_BATCH_SIZE = 2  # chunks per backbone call
USE_GRADIENT_CHECKPOINTING = False
TRAIN_FRACTION = 0.20
# Required: choose YOUR trained decoder-only LoRMA CJPE teacher.
TEACHER_TAG = 'cjpe_decoder_126M_lorma_len2048_ov64_sample2_eval4_best'   # e.g. paste the exact best run tag, including _best
TEACHER_CHECKPOINT_PATH = None  # optional local file or bucket-relative checkpoint path
TEACHER_FALLBACK_ADAPTER_CONFIG = dict(r=16, alpha=32, dropout=0.05, do_pi=True)
TEACHER_CACHE_MAX_CHUNKS = 4
STUDENT_NUM_LAYERS = 4
STUDENT_TRAIN_LAST_N_BLOCKS = 2
COPY_TEACHER_HEAD = True
KD_ALPHA = 0.5  # soft-target loss weight
KD_TEMPERATURE = 2.0
TRAIN_MAX_CHUNKS = 2
EVAL_MAX_CHUNKS = 4
USE_AMP = DEVICE == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16

def autocast_context():
    return torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP)

DATA_PREFIX = 'Phase_4_CJPE'
LABEL_KEY = 'label'  # set to the actual binary label field if the released schema differs
assert 0 < MAX_SEQ_LEN <= MODEL_CONFIG['max_seq_len']
assert 0 <= CHUNK_OVERLAP < MAX_SEQ_LEN
assert CHUNK_ENCODE_BATCH_SIZE > 0
assert 0 <= KD_ALPHA <= 1 and KD_TEMPERATURE > 0
assert 1 <= STUDENT_TRAIN_LAST_N_BLOCKS <= STUDENT_NUM_LAYERS < MODEL_CONFIG['n_layers']
BATCH_SIZE = 1  # one record, at most two training chunks per backward
GRAD_ACCUM_STEPS = 16
LR = 1e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NUM_EPOCHS = 8
PATIENCE = 5
MIN_DELTA = 0.001
HEAD_DROPOUT = 0.1
UPLOAD_CHECKPOINTS = True
TECHNIQUE = 'distillation'

# All loading/saving locations are controlled here.
# Paths below are relative to BUCKET_ROOT; an absolute hf:// path also works.
SAVE_TAG = f'cjpe_{PHASE1_TAG}_{TECHNIQUE}_student{STUDENT_NUM_LAYERS}'  # None generates a descriptive tag; or enter your own run tag.
SUBMISSION_ZIP_NAME = None  # None uses SAVE_TAG + '.zip'; or enter a filename.
CHECKPOINT_PREFIX = 'checkpoints'
RESUME = True  # True resumes SAVE_TAG; set SAVE_TAG to the existing latest tag.
WORK_ROOT = Path('/kaggle/working/rr_runs') if Path('/kaggle/working').exists() else Path.cwd() / 'cjpe_runs'

if RESUME and SAVE_TAG is None:
    raise ValueError('To resume, set SAVE_TAG to the existing run tag.')
if SAVE_TAG is None:
    SAVE_TAG = f'cjpe_{PHASE1_TAG}_{TECHNIQUE}_student{STUDENT_NUM_LAYERS}'
if not isinstance(SAVE_TAG, str) or not re.fullmatch(r'[A-Za-z0-9_.-]+', SAVE_TAG) or SAVE_TAG in {'.', '..'}:
    raise ValueError('SAVE_TAG must be a single name containing letters, numbers, _, - or .')
BEST_TAG = SAVE_TAG + '_best'
if SUBMISSION_ZIP_NAME is None:
    SUBMISSION_ZIP_NAME = SAVE_TAG + '.zip'
if Path(SUBMISSION_ZIP_NAME).name != SUBMISSION_ZIP_NAME or not SUBMISSION_ZIP_NAME.endswith('.zip'):
    raise ValueError('SUBMISSION_ZIP_NAME must be a filename ending in .zip.')
WORK_DIR = WORK_ROOT / SAVE_TAG
WORK_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF')
    except Exception:
        pass
fs = HfFileSystem(token=HF_TOKEN)

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
print('Device:', DEVICE)
def bucket_path(path):
    if path.startswith('hf://'):
        return path
    return BUCKET_ROOT.rstrip('/') + '/' + path.lstrip('/')

def checkpoint_remote_path(tag):
    return bucket_path(f"{CHECKPOINT_PREFIX.strip('/')}/{tag}/checkpoint.pt")

def download_bucket_file(remote_path):
    # Identity-specific cache directory; refresh the bytes on each call.
    remote = bucket_path(remote_path)
    cache_key = hashlib.sha256(remote.encode()).hexdigest()[:20]
    local = WORK_ROOT / '_inputs' / cache_key / remote.rsplit('/', 1)[-1]
    local.parent.mkdir(parents=True, exist_ok=True)
    temporary = local.with_name(local.name + '.download')
    fs.get_file(remote, str(temporary))
    os.replace(temporary, local)
    return local

def assert_run_destination():
    if RESUME:
        return
    for tag in [SAVE_TAG, BEST_TAG]:
        local = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
        if local.exists() or (UPLOAD_CHECKPOINTS and fs.exists(checkpoint_remote_path(tag))):
            raise FileExistsError(
                f'Checkpoint already exists for {tag}. Choose a new SAVE_TAG, '
                'or set RESUME=True with the original latest SAVE_TAG.'
            )
    manifest = dict(save_tag=SAVE_TAG, best_tag=BEST_TAG, technique=TECHNIQUE,
                    phase1_tag=PHASE1_TAG, model_path=MODEL_PATH,
                    tokenizer_path=TOKENIZER_PATH, submission_zip=SUBMISSION_ZIP_NAME,
                    train_fraction=TRAIN_FRACTION, student_layers=STUDENT_NUM_LAYERS,
                    kd_alpha=KD_ALPHA, kd_temperature=KD_TEMPERATURE,
                    train_max_chunks=TRAIN_MAX_CHUNKS, eval_max_chunks=EVAL_MAX_CHUNKS,
                    use_amp=USE_AMP, amp_dtype=str(AMP_DTYPE),
                    max_seq_len=MAX_SEQ_LEN, chunk_overlap=CHUNK_OVERLAP, input_format='overlapping_chunks_v1',
                    pooling='token_mean_then_chunk_mean',
                    student_train_last_n_blocks=STUDENT_TRAIN_LAST_N_BLOCKS)
    with open(WORK_DIR / 'run_config.json', 'w') as f:
        json.dump(manifest, f, indent=2)

print('Phase-1 model:', bucket_path(MODEL_PATH))
print('Tokenizer:', bucket_path(TOKENIZER_PATH))
print('Latest checkpoint:', checkpoint_remote_path(SAVE_TAG))
print('Best checkpoint:', checkpoint_remote_path(BEST_TAG))
print('Submission ZIP:', WORK_DIR / SUBMISSION_ZIP_NAME)

print('Chunk length/overlap:', MAX_SEQ_LEN, CHUNK_OVERLAP)


Device: cuda
Phase-1 model: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/decoder_126M/model.pt
Tokenizer: hf://buckets/A-240031/legal-lm-phase1-bucket/tokenizer_126M_decoder/tokenizer.json
Latest checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/cjpe_decoder_126M_distillation_student4/checkpoint.pt
Best checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/cjpe_decoder_126M_distillation_student4_best/checkpoint.pt
Submission ZIP: /kaggle/working/rr_runs/cjpe_decoder_126M_distillation_student4/cjpe_decoder_126M_distillation_student4.zip
Chunk length/overlap: 2048 64


In [3]:
local_tokenizer = download_bucket_file(TOKENIZER_PATH)
tokenizer = Tokenizer.from_file(str(local_tokenizer))
tokenizer.no_padding()
tokenizer.no_truncation()
VOCAB_SIZE = tokenizer.get_vocab_size()
EMPTY_ID = tokenizer.token_to_id('<|eos|>')
PAD_ID = tokenizer.token_to_id('<|pad|>')
if EMPTY_ID is None:
    raise ValueError('Expected the Phase-1 <|eos|> token.')
if PAD_ID is None: PAD_ID = EMPTY_ID
LABELS = [0, 1]

def download_json(filename):
    path = hf_hub_download(repo_id=DATA_REPO, filename=filename, repo_type='dataset', token=HF_TOKEN)
    with open(path, encoding='utf-8') as f: return json.load(f)

def normalize_documents(raw, labelled):
    if isinstance(raw, list): docs = raw
    elif isinstance(raw, dict) and 'id' in raw and 'text' in raw: docs = [raw]
    elif isinstance(raw, dict):
        docs = []
        for doc_id, value in raw.items():
            if not isinstance(value, dict): raise ValueError('Expected ID -> case object mapping')
            doc = dict(value)
            doc.setdefault('id', doc_id)
            if doc['id'] != doc_id: raise ValueError('Document ID mismatch')
            docs.append(doc)
    else: raise ValueError('Expected a list of cases or ID -> case mapping')
    result = []
    for row_index, doc in enumerate(docs):
        case_id, text = doc['id'], doc['text']
        if not isinstance(text, str): raise ValueError(f'{case_id}: text must be a string')
        row = dict(id=case_id, text=text, row_index=row_index)
        if labelled:
            if LABEL_KEY not in doc:
                raise ValueError(f'Missing label field {LABEL_KEY!r}; available fields: {list(doc)}. Set LABEL_KEY.')
            label = doc[LABEL_KEY]
            if type(label) is not int or label not in (0, 1):
                raise ValueError(f'{case_id}: expected integer label 0/1, got {label!r}')
            row['label'] = label
        result.append(row)
    if not result: raise ValueError('Empty split')
    return result

# Keep each input record separate, even when IDs repeat.
# Test is loaded later, only for prediction.
data = {}
for split in ['train', 'dev']:
    data[split] = normalize_documents(
        download_json(f'{DATA_PREFIX}/Train/{split}.json'), labelled=True)
    counts = Counter(d['id'] for d in data[split])
    print(split, '| records:', len(data[split]), '| unique IDs:', len(counts),
          '| repeated IDs:', sum(n > 1 for n in counts.values()))
    print('Class counts:', dict(Counter(d['label'] for d in data[split])))

# Subset by case ID; repeated records remain separate examples.
# Reloading this cell always starts from the complete downloaded split.
full_train = data['train']
case_labels = {}
for doc in full_train:
    case_labels.setdefault(doc['id'], doc['label'])
    if case_labels[doc['id']] != doc['label']:
        raise ValueError(f"Conflicting labels for training ID {doc['id']}")
case_ids = list(case_labels)
if not 0 < TRAIN_FRACTION <= 1:
    raise ValueError('TRAIN_FRACTION must be in (0, 1]')
if TRAIN_FRACTION < 1:
    chosen, _ = train_test_split(
        case_ids, train_size=TRAIN_FRACTION, random_state=SEED,
        stratify=[case_labels[k] for k in case_ids])
    chosen = set(chosen)
    data['train'] = [doc for doc in full_train if doc['id'] in chosen]
SUBSET_DIGEST = hashlib.sha256(
    json.dumps([d['row_index'] for d in data['train']]).encode()).hexdigest()
DATA_DIGEST = hashlib.sha256(json.dumps(
    {split: docs for split, docs in data.items()},
    sort_keys=True, ensure_ascii=False).encode()).hexdigest()
with open(WORK_DIR / 'train_subset.json', 'w') as f:
    json.dump(dict(fraction=TRAIN_FRACTION, seed=SEED, digest=SUBSET_DIGEST,
                   row_indices=[d['row_index'] for d in data['train']],
                   ids=[d['id'] for d in data['train']]), f)
print('Selected training records:', len(full_train), '→', len(data['train']))
print('Subset classes:', dict(Counter(d['label'] for d in data['train'])))
del full_train


hf://buckets/A-240031/legal-lm-phase1-bu(…): reconstructing file:   0%|          |  0.00B / 2.47MB            

hf://buckets/A-240031/legal-lm-phase1-bu(…): downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

train | records: 37387 | unique IDs: 32305 | repeated IDs: 5082
Class counts: {0: 22067, 1: 15320}


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 80.3MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

dev | records: 3505 | unique IDs: 2511 | repeated IDs: 994
Class counts: {0: 1749, 1: 1756}
Selected training records: 37387 → 7463
Subset classes: {0: 4393, 1: 3070}


In [4]:
def chunk_token_ids(tokens, max_len=MAX_SEQ_LEN, overlap=CHUNK_OVERLAP):
    if not 0 <= overlap < max_len: raise ValueError('Invalid chunk overlap')
    if not tokens: return [[EMPTY_ID]]
    chunks = []
    for start in range(0, len(tokens), max_len - overlap):
        chunks.append(tokens[start:start + max_len])
        if start + max_len >= len(tokens): break
    return chunks

def select_chunk_indices(count, limit, training, seed):
    if limit is None or count <= limit:
        return list(range(count))
    if limit < 1: raise ValueError('Chunk limit must be positive or None')
    if training:
        rng = random.Random(seed)
        # One random chunk from each equal-sized region; two regions = document halves.
        return [rng.randrange(i * count // limit, (i + 1) * count // limit)
                for i in range(limit)]
    if limit == 1: return [count // 2]
    return [i * (count - 1) // (limit - 1) for i in range(limit)]

class CJPECaseDataset(Dataset):
    def __init__(self, documents, split):
        self.split, self.epoch = split, 0
        self.rows = []
        token_lengths, chunk_counts = [], []
        for record_index, doc in enumerate(tqdm(documents, desc='Tokenizing cases')):
            tokens = tokenizer.encode(doc['text'], add_special_tokens=False).ids
            chunks = chunk_token_ids(tokens)
            self.rows.append(dict(doc_id=doc['id'], record_index=record_index,
                                  chunks=chunks, label=doc.get('label', -100)))
            token_lengths.append(len(tokens)); chunk_counts.append(len(chunks))
        lengths = torch.tensor(token_lengths, dtype=torch.float)
        print('Cases/chunks:', len(self.rows), sum(chunk_counts), 'max chunks/case:', max(chunk_counts))
        print('Token lengths p50/p90/p99/max:', torch.quantile(lengths, torch.tensor([.5,.9,.99,1.])).tolist())
    def __len__(self): return len(self.rows)
    def set_epoch(self, epoch): self.epoch = epoch
    def __getitem__(self, index):
        row = self.rows[index]
        training = self.split == 'train'
        limit = TRAIN_MAX_CHUNKS if training else EVAL_MAX_CHUNKS
        selected = select_chunk_indices(len(row['chunks']), limit, training,
                                        SEED + self.epoch * len(self.rows) + index)
        return dict(row, chunks=[row['chunks'][i] for i in selected])

def collate_cjpe(rows):
    chunks, owners = [], []
    for case_index, row in enumerate(rows):
        chunks.extend(row['chunks'])
        owners.extend([case_index] * len(row['chunks']))
    order = sorted(range(len(chunks)), key=lambda i: len(chunks[i]), reverse=True)
    chunks = [chunks[i] for i in order]
    owners = [owners[i] for i in order]
    lengths = torch.tensor([len(c) for c in chunks], dtype=torch.long)
    ids = torch.full((len(chunks), int(lengths.max())), PAD_ID, dtype=torch.long)
    for i, chunk in enumerate(chunks): ids[i, :len(chunk)] = torch.tensor(chunk)
    return dict(input_ids=ids, lengths=lengths, chunk_to_case=torch.tensor(owners, dtype=torch.long),
                labels=torch.tensor([r['label'] for r in rows], dtype=torch.long),
                doc_ids=[r['doc_id'] for r in rows],
                record_indices=torch.tensor([r['record_index'] for r in rows], dtype=torch.long))

datasets = {split: CJPECaseDataset(docs, split) for split, docs in data.items()}
def make_loader(split):
    return DataLoader(datasets[split], batch_size=BATCH_SIZE, shuffle=(split == 'train'),
                      collate_fn=collate_cjpe, num_workers=0, pin_memory=(DEVICE == 'cuda'))


Tokenizing cases:   0%|          | 0/7463 [00:00<?, ?it/s]

Cases/chunks: 7463 19057 max chunks/case: 235
Token lengths p50/p90/p99/max: [2661.0, 8258.0, 23486.49609375, 465531.0]


Tokenizing cases:   0%|          | 0/3505 [00:00<?, ?it/s]

Cases/chunks: 3505 10656 max chunks/case: 31
Token lengths p50/p90/p99/max: [3877.0, 9470.19921875, 25065.775390625, 60245.0]


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2).float() / head_dim)
        )

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, q, k):
        # q, k: [B, H, T, D]
        T = q.size(2)

        cos = self.cos[:T].to(dtype=q.dtype).unsqueeze(0).unsqueeze(0)
        sin = self.sin[:T].to(dtype=q.dtype).unsqueeze(0).unsqueeze(0)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack(
            [q1 * cos - q2 * sin,
             q1 * sin + q2 * cos],
            dim=-1,
        ).flatten(-2)

        k_rot = torch.stack(
            [k1 * cos - k2 * sin,
             k1 * sin + k2 * cos],
            dim=-1,
        ).flatten(-2)

        return q_rot, k_rot


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(
            F.silu(self.gate_proj(x)) * self.up_proj(x)
        )


# ============================================================
# GQA + SDPA
# ============================================================

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len=2048):
        super().__init__()

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

        self.rope = RotaryEmbedding(
            self.head_dim,
            max_seq_len=max_seq_len,
        )

    def forward(self, x):

        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        q, k = self.rope(q, k)
        q = q.to(v.dtype)
        k = k.to(v.dtype)

        # Repeat KV heads for GQA
        repeat_factor = self.n_heads // self.n_kv_heads

        k = k.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        # PyTorch SDPA can dispatch to Flash Attention
        # / memory-efficient kernels automatically.
        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
        )

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, -1)
        )

        return self.out_proj(attn_out)


# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        d_ff,
        max_seq_len=2048,
    ):
        super().__init__()

        self.norm1 = nn.RMSNorm(d_model)
        self.attention = GQAAttention(
            d_model,
            n_heads,
            n_kv_heads,
            max_seq_len=max_seq_len,
        )

        self.norm2 = nn.RMSNorm(d_model)
        self.ffn = SwiGLU(
            d_model,
            d_ff,
        )

    def forward(self, x):

        x = x + self.attention(
            self.norm1(x)
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# Decoder-only Language Model
# ============================================================

class EfficientDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=2048,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.token_embedding = nn.Embedding(                             
            vocab_size,
            d_model,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                d_ff=d_ff,
                max_seq_len=max_seq_len,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.RMSNorm(d_model)

        # Tied LM head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.Embedding):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.RMSNorm):
            nn.init.ones_(module.weight)

    def forward(self, input_ids):

        x = self.token_embedding(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        return self.lm_head(x)


In [6]:
class LoRMALinear(nn.Module):
    def __init__(self, original_linear, r=16, alpha=32, dropout=0.05, do_pi=True):
        super().__init__()
        self.original_linear = original_linear
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.scaling, self.do_pi = alpha / r, do_pi
        self.dropout = nn.Dropout(dropout)
        for p in original_linear.parameters(): p.requires_grad = False
        W = original_linear.weight
        self.A = nn.Parameter(W.new_empty(r, self.out_features))
        self.B = nn.Parameter(W.new_zeros(self.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def effective_weight(self):
        W = self.original_linear.weight
        delta = self.B @ self.A
        if self.do_pi:
            rows = torch.arange(self.out_features, device=W.device)[:, None]
            cols = torch.arange(self.out_features, device=W.device)[None, :]
            delta = delta.gather(1, (cols - rows) % self.out_features)
        return W + self.scaling * (delta @ W)
    def forward(self, x):
        W = self.original_linear.weight
        W_eff = self.effective_weight()
        return F.linear(x, W, self.original_linear.bias) + F.linear(self.dropout(x), W_eff - W)

from torch.utils.checkpoint import checkpoint as activation_checkpoint

class EfficientDecoderLMForCJPE(EfficientDecoderLM):
    def __init__(self, vocab_size, head_dropout=0.1, **kwargs):
        super().__init__(vocab_size=vocab_size, **kwargs)
        del self.lm_head
        self.cjpe_dropout = nn.Dropout(head_dropout)
        self.cjpe_head = nn.Linear(self.d_model, 2)
        self.frozen_prefix_blocks = 0

    def encode_chunks(self, input_ids, lengths):
        with torch.no_grad():
            x = self.token_embedding(input_ids)
            for i in range(self.frozen_prefix_blocks):
                x = self.blocks[i](x)
        for i in range(self.frozen_prefix_blocks, len(self.blocks)):
            x = self.blocks[i](x)
        x = self.final_norm(x)
        # Right padding cannot affect preceding real tokens under causal attention.
        mask = torch.arange(x.size(1), device=x.device)[None, :] < lengths[:, None]
        return (x * mask.unsqueeze(-1)).sum(1, dtype=torch.float32) / lengths[:, None].float()

    def forward(self, input_ids, lengths, chunk_to_case, num_cases):
        device = self.cjpe_head.weight.device
        pooled_chunks = []
        for start in range(0, input_ids.size(0), CHUNK_ENCODE_BATCH_SIZE):
            end = min(start + CHUNK_ENCODE_BATCH_SIZE, input_ids.size(0))
            lens = lengths[start:end].to(device, non_blocking=True)
            # Trim this microbatch to its own longest chunk.
            width = int(lengths[start:end].max())
            ids = input_ids[start:end, :width].to(device, non_blocking=True)
            if self.training and USE_GRADIENT_CHECKPOINTING:
                h = activation_checkpoint(self.encode_chunks, ids, lens, use_reentrant=False,
                                          preserve_rng_state=True)
            else: h = self.encode_chunks(ids, lens)
            pooled_chunks.append(h)
        h = torch.cat(pooled_chunks, dim=0)
        owners = chunk_to_case.to(device, non_blocking=True)
        sums = h.new_zeros((num_cases, self.d_model)).index_add(0, owners, h)
        counts = torch.bincount(owners, minlength=num_cases).to(h.dtype).unsqueeze(1)
        assert bool((counts > 0).all()), 'Every case must have at least one chunk'
        document_vectors = sums / counts
        return self.cjpe_head(self.cjpe_dropout(document_vectors))

def forward_batch(model, batch):
    with autocast_context():
        return model(batch['input_ids'], batch['lengths'], batch['chunk_to_case'], len(batch['doc_ids']))


import time
import copy

# Resolve teacher now so a missing tag fails before expensive training work.
if TEACHER_CHECKPOINT_PATH:
    candidate = Path(TEACHER_CHECKPOINT_PATH)
    teacher_path = candidate if candidate.is_file() else download_bucket_file(TEACHER_CHECKPOINT_PATH)
elif TEACHER_TAG:
    teacher_path = download_bucket_file(checkpoint_remote_path(TEACHER_TAG))
else:
    raise ValueError('Set TEACHER_TAG to your trained decoder-only LoRMA CJPE best checkpoint tag, '
                     'or set TEACHER_CHECKPOINT_PATH. Do not use a Phase-1 LM or RR checkpoint here.')

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for part in iter(lambda: f.read(8 * 1024 * 1024), b''):
            digest.update(part)
    return digest.hexdigest()

TEACHER_SHA256 = file_sha256(teacher_path)
phase1_path = download_bucket_file(MODEL_PATH)
PHASE1_SHA256 = file_sha256(phase1_path)
TOKENIZER_SHA256 = file_sha256(local_tokenizer)
STUDENT_LAYER_INDICES = [round(i * (MODEL_CONFIG['n_layers'] - 1) / (STUDENT_NUM_LAYERS - 1))
                         for i in range(STUDENT_NUM_LAYERS)] if STUDENT_NUM_LAYERS > 1 else [MODEL_CONFIG['n_layers'] - 1]
STUDENT_MODEL_CONFIG = dict(MODEL_CONFIG, n_layers=STUDENT_NUM_LAYERS)
TEACHER_MODEL_INFO = {}
TEACHER_HEAD_STATE = None

def checkpoint_weights(checkpoint):
    for name in ['model_state_dict', 'model', 'state_dict']:
        if name in checkpoint and isinstance(checkpoint[name], dict):
            return dict(checkpoint[name])
    if all(isinstance(v, torch.Tensor) for v in checkpoint.values()): return dict(checkpoint)
    raise ValueError(f'Cannot identify model weights: {list(checkpoint)}')

def load_teacher():
    global TEACHER_HEAD_STATE, TEACHER_MODEL_INFO
    payload = torch.load(teacher_path, map_location='cpu', weights_only=False)
    state = checkpoint_weights(payload)
    metadata = payload.get('metadata', {})
    config = metadata.get('model_config', MODEL_CONFIG)
    shape_fields = ['d_model', 'n_heads', 'n_kv_heads', 'n_layers', 'd_ff']
    if any(config.get(k) != MODEL_CONFIG.get(k) for k in shape_fields):
        raise ValueError('Teacher architecture differs from MODEL_CONFIG. Use its matching dense Phase-1 configuration.')
    if metadata.get('tokenizer_path', TOKENIZER_PATH) != TOKENIZER_PATH:
        raise ValueError('Teacher tokenizer differs from TOKENIZER_PATH; teacher and student must share token IDs.')
    if metadata.get('pooling', 'token_mean_then_chunk_mean') != 'token_mean_then_chunk_mean':
        raise ValueError('This teacher loader expects the CJPE token-mean then chunk-mean wrapper.')
    if metadata.get('labels', [0, 1]) != [0, 1]:
        raise ValueError('Expected binary CJPE teacher labels [0,1].')
    teacher = EfficientDecoderLMForCJPE(VOCAB_SIZE, head_dropout=HEAD_DROPOUT, **MODEL_CONFIG)
    adapter = dict(metadata.get('adapter_config', TEACHER_FALLBACK_ADAPTER_CONFIG))
    # Reconstruct exactly the wrapped projection locations found in the checkpoint,
    # whether adapters were installed in 2 blocks or all blocks.
    wrapped = 0
    for index, block in enumerate(teacher.blocks):
        for name in ['q_proj', 'k_proj', 'v_proj', 'out_proj']:
            prefix = f'blocks.{index}.attention.{name}'
            if prefix + '.original_linear.weight' in state:
                local_config = dict(adapter, r=state[prefix + '.A'].size(0))
                setattr(block.attention, name, LoRMALinear(getattr(block.attention, name), **local_config))
                wrapped += 1
    if not wrapped:
        raise ValueError('No LoRMA projection weights found. Select the decoder-only LoRMA CJPE checkpoint.')
    teacher.load_state_dict(state, strict=True)
    TEACHER_HEAD_STATE = {name: value.detach().cpu().clone() for name, value in teacher.cjpe_head.state_dict().items()}
    original_params = sum(p.numel() for p in teacher.parameters())
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad = False
    # Evaluation dropout is identity, so merge once to avoid reconstructing weights per call.
    with torch.no_grad():
        for block in teacher.blocks:
            for name in ['q_proj', 'k_proj', 'v_proj', 'out_proj']:
                layer = getattr(block.attention, name)
                if isinstance(layer, LoRMALinear):
                    merged = nn.Linear(layer.in_features, layer.out_features,
                                       bias=layer.original_linear.bias is not None)
                    merged.weight.copy_(layer.effective_weight())
                    if merged.bias is not None: merged.bias.copy_(layer.original_linear.bias)
                    setattr(block.attention, name, merged)
    for p in teacher.parameters(): p.requires_grad = False
    teacher.frozen_prefix_blocks = len(teacher.blocks)
    TEACHER_MODEL_INFO = dict(original_params=original_params,
                             merged_params=sum(p.numel() for p in teacher.parameters()),
                             wrapped_projections=wrapped,
                             saved_dev_metrics=payload.get('extra', {}).get('best_f1'))
    del payload, state
    print('Teacher:', TEACHER_MODEL_INFO)
    return teacher.to(DEVICE)

def build_cjpe_model():
    student = EfficientDecoderLMForCJPE(VOCAB_SIZE, head_dropout=HEAD_DROPOUT, **STUDENT_MODEL_CONFIG)
    payload = torch.load(phase1_path, map_location='cpu', weights_only=False)
    source = checkpoint_weights(payload)
    selected = {name: tensor for name, tensor in source.items()
                if name.startswith('token_embedding.') or name.startswith('final_norm.')}
    for new_index, old_index in enumerate(STUDENT_LAYER_INDICES):
        prefix = f'blocks.{old_index}.'
        for name, tensor in source.items():
            if name.startswith(prefix):
                selected[f'blocks.{new_index}.' + name[len(prefix):]] = tensor
    missing, unexpected = student.load_state_dict(selected, strict=False)
    assert set(missing) == {'cjpe_head.weight', 'cjpe_head.bias'} and not unexpected, (missing, unexpected)
    if COPY_TEACHER_HEAD:
        if TEACHER_HEAD_STATE is None: raise RuntimeError('Prepare the teacher cache before building the student.')
        student.cjpe_head.load_state_dict(TEACHER_HEAD_STATE)
    del source, selected, payload
    for p in student.parameters(): p.requires_grad = False
    student.frozen_prefix_blocks = STUDENT_NUM_LAYERS - STUDENT_TRAIN_LAST_N_BLOCKS
    for block in student.blocks[student.frozen_prefix_blocks:]:
        for p in block.parameters(): p.requires_grad = True
    for p in student.final_norm.parameters(): p.requires_grad = True
    for p in student.cjpe_head.parameters(): p.requires_grad = True
    print('Student Phase-1 layer indices:', STUDENT_LAYER_INDICES)
    print('Student total/trainable parameters:', sum(p.numel() for p in student.parameters()),
          sum(p.numel() for p in student.parameters() if p.requires_grad))
    return student.to(DEVICE)

def distillation_loss(student_logits, teacher_logits, labels):
    student_logits, teacher_logits = student_logits.float(), teacher_logits.float()
    ce = F.cross_entropy(student_logits, labels, reduction='sum')
    t = KD_TEMPERATURE
    kl = F.kl_div(F.log_softmax(student_logits / t, dim=-1),
                  F.softmax(teacher_logits / t, dim=-1), reduction='sum') * (t * t)
    return (1 - KD_ALPHA) * ce + KD_ALPHA * kl, ce, kl


hf://buckets/A-240031/legal-lm-phase1-bu(…): reconstructing file:   0%|          |  0.00B /  517MB            

hf://buckets/A-240031/legal-lm-phase1-bu(…): downloading bytes:           |  0.00B            

hf://buckets/A-240031/legal-lm-phase1-bu(…): reconstructing file:   0%|          |  0.00B /  515MB            

hf://buckets/A-240031/legal-lm-phase1-bu(…): downloading bytes:           |  0.00B            

In [7]:
def experiment_metadata(domain='CJPE'):
    return dict(domain=domain, technique='knowledge_distillation', phase1_tag=PHASE1_TAG,
                model_path=MODEL_PATH, phase1_sha256=PHASE1_SHA256, tokenizer_path=TOKENIZER_PATH,
                labels=LABELS, teacher_sha256=TEACHER_SHA256, tokenizer_sha256=TOKENIZER_SHA256,
                teacher_fallback_adapter_config=TEACHER_FALLBACK_ADAPTER_CONFIG,
                teacher_model_config=MODEL_CONFIG, student_model_config=STUDENT_MODEL_CONFIG,
                student_layer_indices=STUDENT_LAYER_INDICES,
                student_train_last_n_blocks=STUDENT_TRAIN_LAST_N_BLOCKS,
                copy_teacher_head=COPY_TEACHER_HEAD, kd_alpha=KD_ALPHA, temperature=KD_TEMPERATURE,
                teacher_cache_max_chunks=TEACHER_CACHE_MAX_CHUNKS,
                train_fraction=TRAIN_FRACTION, subset_digest=SUBSET_DIGEST, data_digest=DATA_DIGEST,
                max_seq_len=MAX_SEQ_LEN, chunk_overlap=CHUNK_OVERLAP,
                chunk_encode_batch_size=CHUNK_ENCODE_BATCH_SIZE,
                gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
                train_max_chunks=TRAIN_MAX_CHUNKS, eval_max_chunks=EVAL_MAX_CHUNKS,
                sampling='stratified_epoch_seed_v1', use_amp=USE_AMP, amp_dtype=str(AMP_DTYPE),
                pooling='token_mean_then_chunk_mean', seed=SEED, head_dropout=HEAD_DROPOUT,
                data_repo=DATA_REPO, data_prefix=DATA_PREFIX, label_key=LABEL_KEY,
                batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
                lr=LR, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
                patience=PATIENCE, min_delta=MIN_DELTA)

def save_cjpe_checkpoint(tag, model, optimizer, epoch, extra, domain):
    folder = WORK_DIR / 'checkpoints' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rng = {'python': random.getstate(), 'torch': torch.get_rng_state()}
    if torch.cuda.is_available(): rng['cuda'] = torch.cuda.get_rng_state_all()
    payload = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), rng=rng,
                   epoch=epoch, extra=extra, metadata=experiment_metadata(domain))
    tmp = folder / 'checkpoint.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, folder / 'checkpoint.pt')
    if UPLOAD_CHECKPOINTS:
        fs.put_file(str(folder / 'checkpoint.pt'), checkpoint_remote_path(tag))

def load_cjpe_checkpoint(tag, model, domain, optimizer=None, restore_rng=False):
    path = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
    if not path.exists() or restore_rng:
        if UPLOAD_CHECKPOINTS:
            path = download_bucket_file(checkpoint_remote_path(tag))
        elif not path.exists():
            raise FileNotFoundError(path)
    payload = torch.load(path, map_location='cpu', weights_only=False)  # own trusted checkpoints only
    assert payload['metadata'] == experiment_metadata(domain), 'Checkpoint configuration mismatch'
    model.load_state_dict(payload['model'])
    if optimizer is not None: optimizer.load_state_dict(payload['optimizer'])
    if restore_rng:
        random.setstate(payload['rng']['python'])
        torch.set_rng_state(payload['rng']['torch'])
        if torch.cuda.is_available() and 'cuda' in payload['rng']:
            torch.cuda.set_rng_state_all(payload['rng']['cuda'])
    return payload

@torch.no_grad()
def evaluate_cjpe(model, loader):
    model.eval()
    targets, predictions = [], []
    total_loss = total_n = 0
    for batch in loader:
        y = batch['labels'].to(DEVICE)
        logits = forward_batch(model, batch)
        total_loss += F.cross_entropy(logits.float(), y, reduction='sum').item()
        total_n += len(y)
        targets.extend(y.cpu().tolist())
        predictions.extend(logits.argmax(-1).cpu().tolist())
    assert total_n > 0, 'Development set has no cases'
    p, r, f, support = precision_recall_fscore_support(targets, predictions, labels=[0, 1], zero_division=0)
    return dict(loss=total_loss / total_n, macro_f1=float(f.mean()), macro_precision=float(p.mean()),
                macro_recall=float(r.mean()), per_label={label: dict(precision=float(p[i]), recall=float(r[i]),
                f1=float(f[i]), support=int(support[i])) for i, label in enumerate(LABELS)},
                confusion_matrix=confusion_matrix(targets, predictions, labels=[0, 1]).tolist())


In [8]:
class TeacherChunkView(Dataset):
    def __init__(self, dataset): self.rows = dataset.rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        chosen = select_chunk_indices(len(row['chunks']), TEACHER_CACHE_MAX_CHUNKS, False, SEED)
        return dict(row, chunks=[row['chunks'][i] for i in chosen])

def teacher_cache_metadata():
    return dict(teacher_sha256=TEACHER_SHA256, data_digest=DATA_DIGEST,
                tokenizer_sha256=TOKENIZER_SHA256, fallback_adapter=TEACHER_FALLBACK_ADAPTER_CONFIG,
                subset_digest=SUBSET_DIGEST, tokenizer_path=TOKENIZER_PATH,
                max_seq_len=MAX_SEQ_LEN, overlap=CHUNK_OVERLAP,
                teacher_chunks=TEACHER_CACHE_MAX_CHUNKS, pooling='token_mean_then_chunk_mean',
                amp=USE_AMP, dtype=str(AMP_DTYPE))

@torch.no_grad()
def prepare_teacher_cache():
    global TEACHER_HEAD_STATE, TEACHER_MODEL_INFO
    cache_path = WORK_DIR / 'teacher_train_logits.pt'
    if cache_path.exists():
        saved = torch.load(cache_path, map_location='cpu', weights_only=False)
        if saved['metadata'] == teacher_cache_metadata():
            if saved['logits'].shape != (len(datasets['train']), 2):
                raise ValueError('Teacher cache shape mismatch')
            TEACHER_HEAD_STATE, TEACHER_MODEL_INFO = saved['head'], saved['teacher_info']
            print('Reusing train-only teacher logits:', cache_path)
            return saved['logits'].float()
        print('Teacher/data/coverage changed; rebuilding the local cache.')
    teacher = load_teacher()
    teacher.eval()
    train_logits = torch.empty(len(datasets['train']), 2, dtype=torch.float32)
    filled = torch.zeros(len(datasets['train']), dtype=torch.bool)
    loader = DataLoader(TeacherChunkView(datasets['train']), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate_cjpe, num_workers=0)
    begin = time.perf_counter()
    for batch in tqdm(loader, desc='Cache teacher logits (one pass)'):
        positions = batch['record_indices']
        train_logits[positions] = forward_batch(teacher, batch).float().cpu()
        filled[positions] = True
    if not filled.all() or not torch.isfinite(train_logits).all():
        raise ValueError('Teacher cache has missing/nonfinite logits')
    teacher_metrics = evaluate_cjpe(teacher, make_loader('dev'))
    teacher_info = dict(TEACHER_MODEL_INFO, dev_metrics=teacher_metrics,
                        cache_seconds=time.perf_counter() - begin,
                        dev_max_chunks=EVAL_MAX_CHUNKS)
    TEACHER_MODEL_INFO = teacher_info
    temp = cache_path.with_suffix('.tmp')
    torch.save(dict(metadata=teacher_cache_metadata(), logits=train_logits,
                    head=TEACHER_HEAD_STATE, teacher_info=teacher_info), temp)
    os.replace(temp, cache_path)
    with open(WORK_DIR / 'teacher_dev_metrics.json', 'w') as f: json.dump(teacher_info, f, indent=2)
    print('Teacher dev Macro-F1:', teacher_metrics['macro_f1'])
    del teacher
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return train_logits

# One frozen-teacher pass; no teacher forward is needed in student epochs.
TEACHER_TRAIN_LOGITS = prepare_teacher_cache()


Teacher: {'original_params': 128858114, 'merged_params': 128661506, 'wrapped_projections': 8, 'saved_dev_metrics': 0.6201148966237597}


Cache teacher logits (one pass):   0%|          | 0/7463 [00:00<?, ?it/s]

Teacher dev Macro-F1: 0.620654244344415


In [9]:
def train_cjpe(domain='CJPE'):
    assert domain == 'CJPE'
    assert_run_destination()
    seed_everything(SEED)
    model = build_cjpe_model()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16)
    tag = SAVE_TAG
    best_tag = BEST_TAG
    start = 0
    extra = dict(best_f1=-1.0, patience_reference=-1.0, stale=0, history=[], best_tag=best_tag, global_step=0)
    if RESUME:
        checkpoint = load_cjpe_checkpoint(SAVE_TAG, model, domain, optimizer, restore_rng=True)
        start, extra = checkpoint['epoch'], checkpoint['extra']
        best_tag = extra['best_tag']
        if 'scaler' in extra: scaler.load_state_dict(extra['scaler'])
        del checkpoint
    train_loader, dev_loader = make_loader('train'), make_loader('dev')
    assert len(train_loader) > 0
    for epoch in range(start, NUM_EPOCHS):
        if extra['stale'] >= PATIENCE: break
        datasets['train'].set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_sum = train_n = group_n = 0
        epoch_ce = epoch_kl = 0.0
        for step, batch in enumerate(tqdm(train_loader, desc=f'{domain} epoch {epoch+1}')):
            y = batch['labels'].to(DEVICE)
            logits = forward_batch(model, batch)
            teacher_logits = TEACHER_TRAIN_LOGITS[batch['record_indices']].to(DEVICE)
            loss_sum, ce_sum, kl_sum = distillation_loss(logits, teacher_logits, y)
            epoch_ce += ce_sum.item()
            epoch_kl += kl_sum.item()
            scaler.scale(loss_sum).backward()
            group_n += len(y)
            train_n += len(y)
            train_sum += loss_sum.item()
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == len(train_loader):
                scaler.unscale_(optimizer)
                for p in params:
                    if p.grad is not None: p.grad.div_(group_n)
                torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                successful_update = (not scaler.is_enabled() or scaler.get_scale() >= previous_scale)
                if successful_update:
                    extra['global_step'] += 1
                optimizer.zero_grad(set_to_none=True)
                group_n = 0
        metrics = evaluate_cjpe(model, dev_loader)
        f1 = metrics['macro_f1']
        improved = f1 > extra['best_f1']
        if improved: extra['best_f1'] = f1
        if f1 > extra['patience_reference'] + MIN_DELTA:
            extra['patience_reference'], extra['stale'] = f1, 0
        else: extra['stale'] += 1
        extra['history'].append(dict(epoch=epoch+1, train_loss=train_sum/train_n,
                                     global_step=extra['global_step'], train_ce=epoch_ce/train_n,
                                     train_kl=epoch_kl/train_n, **metrics))
        print(f"CJPE train loss={train_sum/train_n:.4f}, dev macro-F1={f1:.4f}, precision={metrics['macro_precision']:.4f}, recall={metrics['macro_recall']:.4f}, patience={extra['stale']}/{PATIENCE}")
        extra['scaler'] = scaler.state_dict()
        if improved: save_cjpe_checkpoint(best_tag, model, optimizer, epoch+1, extra, domain)
        save_cjpe_checkpoint(tag, model, optimizer, epoch+1, extra, domain)
    load_cjpe_checkpoint(best_tag, model, domain)
    metrics = evaluate_cjpe(model, dev_loader)
    with open(WORK_DIR / f'{domain}_dev_metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
    with open(WORK_DIR / f'{domain}_history.json', 'w') as f: json.dump(extra['history'], f, indent=2)
    teacher_params = TEACHER_MODEL_INFO.get('merged_params')
    student_params = sum(p.numel() for p in model.parameters())
    comparison = dict(teacher=TEACHER_MODEL_INFO, student_params=student_params,
                      student_trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad),
                      student_dev_metrics=metrics,
                      parameter_reduction_fraction=(1 - student_params/teacher_params) if teacher_params else None)
    with open(WORK_DIR / 'distillation_comparison.json', 'w') as f: json.dump(comparison, f, indent=2)
    print('Best student development metrics:', metrics)
    del optimizer
    return model, extra['history']


In [10]:
def validate_predictions(predictions, documents):
    assert len(predictions) == len(documents)
    assert [p['id'] for p in predictions] == [d['id'] for d in documents]
    assert len({p['id'] for p in predictions}) == len(predictions)
    assert all(set(p) == {'id', 'pred'} and type(p['pred']) is int and p['pred'] in (0, 1) for p in predictions)

@torch.no_grad()
def predict_cjpe(model):
    model.eval()
    predictions = []
    for batch in tqdm(make_loader('test'), desc='CJPE test predictions'):
        labels = forward_batch(model, batch).argmax(-1).cpu().tolist()
        predictions.extend(dict(id=doc_id, pred=int(label)) for doc_id, label in zip(batch['doc_ids'], labels))
    validate_predictions(predictions, data['test'])
    return predictions

cjpe_model, cjpe_history = train_cjpe()
# Load test only after training. Its IDs are unique in the supplied data.
data['test'] = normalize_documents(
    download_json(f'{DATA_PREFIX}/Test/test.json'), labelled=False)
datasets['test'] = CJPECaseDataset(data['test'], 'test')
predictions = predict_cjpe(cjpe_model)
json_path = WORK_DIR / 'prediction.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
del cjpe_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


Student Phase-1 layer indices: [0, 2, 5, 7]
Student total/trainable parameters: 81467394 23600130


hf://buckets/A-240031/legal-lm-phase1-bu(…): reconstructing file:   0%|          |  0.00B /  515MB            

hf://buckets/A-240031/legal-lm-phase1-bu(…): downloading bytes:           |  0.00B            

CJPE epoch 8:   0%|          | 0/7463 [00:00<?, ?it/s]

CJPE train loss=0.2003, dev macro-F1=0.6320, precision=0.6320, recall=0.6320, patience=2/5


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (6 outputs left out)

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Tokenizing cases:   0%|          | 0/1517 [00:00<?, ?it/s]

Cases/chunks: 1517 4664 max chunks/case: 29
Token lengths p50/p90/p99/max: [3967.0, 9680.0, 24404.427734375, 56610.0]


CJPE test predictions:   0%|          | 0/1517 [00:00<?, ?it/s]

In [11]:
# Packaging can be rerun without retraining.
json_path = WORK_DIR / 'prediction.json'
with open(json_path, encoding='utf-8') as f: predictions = json.load(f)
validate_predictions(predictions, data['test'])
zip_path = WORK_DIR / SUBMISSION_ZIP_NAME
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname='prediction.json')
print('JSON:', json_path)
print('ZIP:', zip_path)
print('Cases:', len(predictions))


JSON: /kaggle/working/rr_runs/cjpe_decoder_126M_distillation_student4/prediction.json
ZIP: /kaggle/working/rr_runs/cjpe_decoder_126M_distillation_student4/cjpe_decoder_126M_distillation_student4.zip
Cases: 1517
